# Re-implementation Plan: Key Architectures from 'Molecular Representation Learning'

This notebook sketches a plan for implementing the key architectural components discussed in the review paper, "Molecular representation learning: cross-domain foundations and future Frontiers".

The paper is a broad review, so we will focus on planning a single, representative model that synthesizes the user-requested components:
1.  **Node Embedding** (for atoms)
2.  **Spatial Encoding** (for 3D geometry/graph structure)
3.  **Masked Pretraining** (as a self-supervised learning strategy)

A **Graph Transformer** model (e.g., Graphormer-style) is an ideal target as it explicitly uses these three components. We will design the APIs using PyTorch-style signatures.

## 1. Core Component API Sketches

We'll first define the APIs for the three small, independent components.

### 1.1. Component 1: Node Embedding (Atom Embedding)

This module is responsible for converting raw atomic features (like atomic number, charge, hybridization) into a continuous vector representation.

**Inputs:** A batch of graph data, typically including a tensor of atomic features.
**Outputs:** A tensor of node embeddings, `[num_nodes_in_batch, embedding_dim]`.

In [1]:
import torch
from torch import nn

class AtomEncoder(nn.Module):
    """
    Encodes raw atomic features (e.g., atomic number, charge, chirality)
    into a dense vector embedding.
    """
    def __init__(self, embedding_dim: int, num_atom_features: int):
        super().__init__()
        self.embedding_dim = embedding_dim
        # Example: A simple linear layer. In practice, this could use
        # separate nn.Embedding layers for categorical features (like atomic num)
        # and linear layers for continuous features.
        self.atom_embedding_layer = nn.Linear(num_atom_features, embedding_dim)
        print(f"[Init] AtomEncoder: {num_atom_features} features -> {embedding_dim} dim")

    def forward(self, atom_features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            atom_features (torch.Tensor): Tensor of shape [N, F],
                where N is the total number of nodes in the batch,
                and F is the number of raw atom features.

        Returns:
            torch.Tensor: Node embedding tensor of shape [N, E],
                where E is embedding_dim.
        """
        # This is a simplified example. A real implementation would handle
        # different feature types (categorical, continuous) more robustly.
        return self.atom_embedding_layer(atom_features)

NUM_NODES = 50       
NUM_FEATURES = 32   
EMBED_DIM = 128

dummy_atom_features = torch.randn(NUM_NODES, NUM_FEATURES)
atom_encoder = AtomEncoder(embedding_dim=EMBED_DIM, num_atom_features=NUM_FEATURES)
node_embeddings = atom_encoder(dummy_atom_features)

print(f"Input shape:  {dummy_atom_features.shape}")
print(f"Output shape: {node_embeddings.shape}")

[Init] AtomEncoder: 32 features -> 128 dim
Input shape:  torch.Size([50, 32])
Output shape: torch.Size([50, 128])


### 1.2. Component 2: Spatial Encoding

This module encodes the 3D structure or graph topology. For a Graph Transformer, this is often implemented as a **bias** to the attention mechanism. It tells the model how "close" nodes are, either in 3D space (spatial distance) or in the graph (shortest path distance).

**Inputs:** Adjacency matrix, or 3D coordinates.
**Outputs:** An attention bias tensor, `[batch_size, num_nodes, num_nodes]`.

In [ ]:
def get_shortest_path_distances(adj_matrix: torch.Tensor) -> torch.Tensor:
    """
    Helper to compute all-pairs shortest path. (e.g., using Floyd-Warshall)
    This is a placeholder for the actual algorithm.
    """
    num_nodes = adj_matrix.shape[0]
    # In a real implementation, you'd run Floyd-Warshall or batched BFS.
    # Here, we just return a placeholder.
    print("[Helper] Calculating shortest path distances... (placeholder)")
    # A real matrix would have 0 on diagonal, 1 for neighbors, 2+ for paths
    return torch.randint(0, 10, (num_nodes, num_nodes)).float()

class SpatialEncoder(nn.Module):
    """
    Encodes graph spatial/topological information (e.g., shortest path distance)
    into a bias for the self-attention mechanism.
    """
    def __init__(self, max_path_distance: int, num_attention_heads: int):
        super().__init__()
        self.max_path_distance = max_path_distance
        self.num_attention_heads = num_attention_heads
        
        # Learnable embedding for each possible shortest path distance
        # +1 for "unreachable"
        self.distance_embedding = nn.Embedding(
            max_path_distance + 2, num_attention_heads
        )
        print(f"[Init] SpatialEncoder: max_dist={max_path_distance}, heads={num_attention_heads}")

    def forward(self, adj_matrix: torch.Tensor) -> torch.Tensor:
        """
        Args:
            adj_matrix (torch.Tensor): Batch of adjacency matrices,
                shape [B, N, N], where B is batch size, N is num nodes.

        Returns:
            torch.Tensor: The spatial attention bias, shape [B, N, N, H],
                where H is num_attention_heads.
                This can be added directly to the attention logits.
        """
        batch_size, num_nodes, _ = adj_matrix.shape
        
        # 1. Get shortest path distances for each graph in the batch
        # (This is a placeholder; batching this is complex)
        path_distances = []
        for i in range(batch_size):
            dist = get_shortest_path_distances(adj_matrix[i])
            dist.clamp_(0, self.max_path_distance + 1) # Cap distances
            path_distances.append(dist)
        
        path_distances_tensor = torch.stack(path_distances).long()
        
        # 2. Get learnable embeddings for these distances
        # [B, N, N] -> [B, N, N, H]
        spatial_bias = self.distance_embedding(path_distances_tensor)
        
        return spatial_bias

# --- API Shape Example ---
BATCH_SIZE = 4
NUM_NODES = 16 # Max nodes in a graph (padded)
NUM_HEADS = 8
MAX_DIST = 10

dummy_adj = torch.randint(0, 2, (BATCH_SIZE, NUM_NODES, NUM_NODES))
spatial_encoder = SpatialEncoder(max_path_distance=MAX_DIST, num_attention_heads=NUM_HEADS)
attention_bias = spatial_encoder(dummy_adj)

print(f"Input shape:  {dummy_adj.shape}")
print(f"Output shape: {attention_bias.shape}")

### 1.3. Component 3: Masked Pretraining (SSL)

This is a *strategy*, not just one module. It involves two parts:
1.  **Masking Utility:** A function to randomly mask nodes or edges.
2.  **Prediction Head:** A module (usually an MLP) to predict the masked-out features from the model's final hidden state.

**Masking Utility (Inputs/Outputs):**
* **Input:** Batch of graph data.
* **Output:** `(masked_batch, node_labels, edge_labels)`

**Prediction Head (Inputs/Outputs):**
* **Input:** Final node embeddings from the model, `[num_nodes_in_batch, hidden_dim]`.
* **Output:** Predicted logits for masked features, `[num_masked_nodes, feature_vocab_size]`.

In [ ]:
def mask_graph_nodes(atom_features: torch.Tensor, mask_prob: float = 0.15) -> (torch.Tensor, torch.Tensor, torch.Tensor):
    """
    Masks nodes in the graph for masked atom prediction.
    
    Args:
        atom_features (torch.Tensor): The original atom features [N, F].
        mask_prob (float): Probability of masking any given node.

    Returns:
        masked_features (torch.Tensor): Features with some nodes replaced by a [MASK] token.
        mask_indices (torch.Tensor): Boolean tensor indicating which nodes were masked.
        original_labels (torch.Tensor): The original feature values for the masked nodes.
    """
    num_nodes, num_features = atom_features.shape
    
    # Create a mask tensor
    prob = torch.rand(num_nodes)
    mask_indices = prob < mask_prob
    
    masked_features = atom_features.clone()
    
    # Create a [MASK] feature vector (e.g., all zeros or a learnable vector)
    # Here we'll use a simple all-zeros vector for planning.
    mask_token_vector = torch.zeros(num_features)
    
    masked_features[mask_indices] = mask_token_vector
    
    # We need the original labels to calculate the loss
    # This assumes we are predicting the raw features, e.g., atomic number (as a class)
    # Let's assume the 0-th feature is the atomic number index
    original_labels = atom_features[mask_indices, 0].long() # Example
    
    print(f"[Masking] Masked {mask_indices.sum()} out of {num_nodes} nodes.")
    return masked_features, mask_indices, original_labels


class MaskedAtomPredictionHead(nn.Module):
    """
    Predicts the original atom type from the final hidden state of masked nodes.
    """
    def __init__(self, hidden_dim: int, atom_vocab_size: int):
        super().__init__()
        # Simple 2-layer MLP
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, atom_vocab_size)
        )
        print(f"[Init] MaskedAtomPredictionHead: {hidden_dim} dim -> {atom_vocab_size} vocab")

    def forward(self, final_hidden_states: torch.Tensor, mask_indices: torch.Tensor) -> torch.Tensor:
        """
        Args:
            final_hidden_states (torch.Tensor): [N, H] output from the main model.
            mask_indices (torch.Tensor): [N] boolean tensor of masked nodes.

        Returns:
            torch.Tensor: Logits for the masked nodes, [num_masked, vocab_size].
        """
        # Select only the hidden states of the nodes that were masked
        masked_node_states = final_hidden_states[mask_indices]
        
        # Predict the logits
        logits = self.mlp(masked_node_states)
        return logits

# --- API Shape Example ---
HIDDEN_DIM = 128
ATOM_VOCAB_SIZE = 119 # (Periodic table + padding)
dummy_features = torch.randn(NUM_NODES, NUM_FEATURES)
dummy_features[:, 0] = torch.randint(0, ATOM_VOCAB_SIZE, (NUM_NODES,)) # Set atomic num

masked_features, mask_idx, labels = mask_graph_nodes(dummy_features)
head = MaskedAtomPredictionHead(HIDDEN_DIM, ATOM_VOCAB_SIZE)

# This would be the output of the main Graphormer model
dummy_hidden_states = torch.randn(NUM_NODES, HIDDEN_DIM)
logits = head(dummy_hidden_states, mask_idx)

print(f"Input states shape: {dummy_hidden_states.shape}")
print(f"Mask indices shape: {mask_idx.shape}")
print(f"Output logits shape:  {logits.shape} (matches num masked)")
print(f"Labels shape:         {labels.shape} (matches num masked)")

## 2. Synthesizing the Full Model

Now we can sketch the full `Graphormer`-style model that combines these components.

In [ ]:
class GraphormerTransformerLayer(nn.Module):
    """A single layer of the Graph Transformer."""
    def __init__(self, embedding_dim: int, num_heads: int):
        super().__init__()
        self.attention = nn.MultiheadAttention(embedding_dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim * 4),
            nn.ReLU(),
            nn.Linear(embedding_dim * 4, embedding_dim)
        )

    def forward(self, x: torch.Tensor, attn_bias: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (torch.Tensor): Node embeddings [B, N, E].
            attn_bias (torch.Tensor): Spatial bias [B, N, N, H] from SpatialEncoder.
                This needs to be reshaped/broadcasted to [B*H, N, N].
        Returns:
            torch.Tensor: Updated node embeddings [B, N, E].
        """
        batch_size, num_nodes, _ = x.shape
        num_heads = self.attention.num_heads
        
        # 1. Prepare attention bias
        # [B, N, N, H] -> [B, H, N, N] -> [B*H, N, N]
        bias = attn_bias.permute(0, 3, 1, 2).reshape(batch_size * num_heads, num_nodes, num_nodes)
        
        # 2. Multi-head Attention (with spatial bias)
        attn_output, _ = self.attention(
            x, x, x, 
            attn_mask=bias, 
            need_weights=False
        )
        
        # 3. Add & Norm
        x = self.norm1(x + attn_output)
        
        # 4. FFN
        ffn_output = self.ffn(x)
        
        # 5. Add & Norm
        x = self.norm2(x + ffn_output)
        
        return x


class GraphormerForPretraining(nn.Module):
    """
    Full Graphormer-style model for masked pretraining.
    """
    def __init__(self, num_atom_features: int, embedding_dim: int, num_heads: int, 
                 num_layers: int, max_path_distance: int, atom_vocab_size: int):
        super().__init__()
        
        # Component 1: Node Embedding
        self.atom_encoder = AtomEncoder(embedding_dim, num_atom_features)
        
        # Component 2: Spatial Encoding
        self.spatial_encoder = SpatialEncoder(max_path_distance, num_heads)
        
        # Core Transformer Layers
        self.layers = nn.ModuleList(
            [GraphormerTransformerLayer(embedding_dim, num_heads) for _ in range(num_layers)]
        )
        
        # Component 3: Prediction Head
        self.prediction_head = MaskedAtomPredictionHead(embedding_dim, atom_vocab_size)

    def forward(self, atom_features: torch.Tensor, adj_matrix: torch.Tensor) -> torch.Tensor:
        """
        A simplified forward pass for pretraining.
        Note: This API assumes graphs are batched as dense tensors [B, N, F] 
        and [B, N, N]. A real implementation would use a library like
        PyTorch Geometric to handle sparse/variable-sized graphs.
        
        Args:
            atom_features (torch.Tensor): [B, N, F]
            adj_matrix (torch.Tensor): [B, N, N]
            
        Returns:
            torch.Tensor: Final node representations [B, N, E]
        """
        # 0. Get dimensions
        batch_size, num_nodes, num_features = atom_features.shape
        embedding_dim = self.atom_encoder.embedding_dim
        
        # 1. Get Node Embeddings
        # [B, N, F] -> [B*N, F] -> [B*N, E] -> [B, N, E]
        x = atom_features.reshape(-1, num_features)
        x = self.atom_encoder(x)
        x = x.reshape(batch_size, num_nodes, embedding_dim)
        
        # 2. Get Spatial Bias
        # [B, N, N] -> [B, N, N, H]
        spatial_bias = self.spatial_encoder(adj_matrix)
        
        # 3. Pass through Transformer Layers
        for layer in self.layers:
            x = layer(x, spatial_bias)
            
        return x

## 3. Pretraining Pipeline Sketch

Finally, here is a sketch of the `train_step` function that uses these components for the masked pretraining task.

In [ ]:
# --- Model Initialization (Example) ---
model = GraphormerForPretraining(
    num_atom_features=NUM_FEATURES,
    embedding_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=6,
    max_path_distance=MAX_DIST,
    atom_vocab_size=ATOM_VOCAB_SIZE
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# --- Pseudo-code for a single training step ---

def train_step(batch_atom_features, batch_adj_matrix):
    """
    A single step of the pretraining loop.
    Assumes dense batching [B, N, F] and [B, N, N].
    """
    
    # 1. Mask the input nodes
    # We need to do this per-graph in the batch, or adapt the helper
    # For simplicity, let's assume we adapt `mask_graph_nodes` to handle batches
    # and that atom_features[..., 0] is the atomic number index.
    
    # This is a placeholder for a batched masking function:
    # masked_features, mask_idx_flat, labels_flat = batched_mask_graph_nodes(batch_atom_features)
    # --- Placeholder Start ---
    flat_features = batch_atom_features.reshape(-1, NUM_FEATURES)
    masked_features_flat, mask_idx_flat, labels_flat = mask_graph_nodes(flat_features)
    masked_features_batch = masked_features_flat.reshape(BATCH_SIZE, NUM_NODES, NUM_FEATURES)
    # --- Placeholder End ---
    
    
    # 2. Forward pass with the masked graph
    optimizer.zero_grad()
    final_hidden_states = model(masked_features_batch, batch_adj_matrix)
    # final_hidden_states shape is [B, N, E]
    
    # 3. Get predictions for masked nodes
    # Reshape hidden states to [B*N, E] to match flat mask indices
    final_hidden_states_flat = final_hidden_states.reshape(-1, EMBED_DIM)
    
    logits = model.prediction_head(final_hidden_states_flat, mask_idx_flat)
    # logits shape is [num_masked_total, atom_vocab_size]
    
    # 4. Calculate loss
    loss = criterion(logits, labels_flat)
    
    # 5. Backward pass and optimization
    loss.backward()
    optimizer.step()
    
    return loss.item()

# --- API Shape Example ---
dummy_atom_features_batch = torch.randn(BATCH_SIZE, NUM_NODES, NUM_FEATURES)
dummy_atom_features_batch[:, :, 0] = torch.randint(0, ATOM_VOCAB_SIZE, (BATCH_SIZE, NUM_NODES))
dummy_adj_batch = torch.randint(0, 2, (BATCH_SIZE, NUM_NODES, NUM_NODES))

print("--- Running single training step sketch ---")
loss = train_step(dummy_atom_features_batch, dummy_adj_batch)
print(f"[Train Step] Pseudo-code executed. Example Loss: {loss:.4f}")

## 4. Conclusion

This plan outlines the key components for a Graph Transformer pretraining pipeline, inspired by models like **Graphormer** and **GROVER** discussed in the review.

**Key Modules Planned:**
-   `AtomEncoder(nn.Module)`: Handles atom-level feature embedding.
-   `SpatialEncoder(nn.Module)`: Creates a learnable attention bias from graph distances.
-   `mask_graph_nodes(...)`: Utility for the SSL masking task.
-   `MaskedAtomPredictionHead(nn.Module)`: Predicts masked atoms from final embeddings.
-   `GraphormerForPretraining(nn.Module)`: The final model that ties all components together.

**Next Steps:**
1.  **Data Loading:** The main challenge is creating a `DataLoader` that efficiently batches variable-sized graphs into dense, padded tensors (`[B, N, F]` and `[B, N, N]`) required by the Transformer, along with a batch-level adjacency matrix for spatial encoding.
2.  **Helper Functions:** Implement a robust, batched all-pairs shortest path algorithm (e.g., batched Floyd-Warshall or repeated BFS) to replace the `get_shortest_path_distances` placeholder.
3.  **Refine Masking:** The `mask_graph_nodes` utility should be adapted to handle batches and different masking strategies (e.g., masking edges, masking subgraphs) as mentioned in the paper.